In [3]:
import numpy as np
import pandas as pd


def clean_agent_data(txns: pd.DataFrame, agents: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """Clean agent transaction records and validate referential integrity against the agents table.

    Applies deterministic, rule-based cleaning with zero hard-coded row numbers:
      1. Trims whitespace and standardizes `txn_type` casing (e.g., 'DEPOSIT ' -> 'Deposit').
      2. Parses `txn_date` across mixed formats (`YYYY-MM-DD` and `YYYY/MM/DD`) into `datetime64[ns]`.
      3. Deduplicates records first by exact row match and then by primary key (`txn_id`), keeping
         the valid non-orphan agent record when a duplicate `txn_id` has a corrupted `agent_id`.
      4. Coerces `amount` to numeric (`NaN` for invalid strings like 'N/A'), drops missing or
         non-positive amounts, and removes extreme log-scale outliers (modified Z-score > 4.0).
      5. Enforces referential integrity by filtering out transactions whose `agent_id` does not
         exist in `agents['agent_id']`, then joins agent metadata (`county`, `onboarded`).

    Args:
        txns: Raw agent transactions DataFrame.
        agents: Master agent lookup DataFrame containing `agent_id`, `county`, and `onboarded`.

    Returns:
        A tuple `(clean_df, log)` where `clean_df` is the validated, enriched DataFrame and `log`
        is an audit dictionary summarizing row counts and anomalies resolved at each step.
    """
    log: dict = {
        "initial_txn_rows": int(len(txns)),
        "initial_agent_rows": int(len(agents)),
    }

    df = txns.copy()
    agents_clean = agents.copy()

    # 1. Standardize identifiers, categorical strings, and agent onboarding dates
    agents_clean["agent_id"] = agents_clean["agent_id"].astype(str).str.strip().str.upper()
    agents_clean["county"] = agents_clean["county"].astype(str).str.strip().str.title()
    agents_clean["onboarded"] = pd.to_datetime(agents_clean["onboarded"], format="mixed")

    df["txn_id"] = df["txn_id"].astype(str).str.strip().str.upper()
    df["agent_id"] = df["agent_id"].astype(str).str.strip().str.upper()

    raw_types = df["txn_type"].astype(str)
    df["txn_type"] = raw_types.str.strip().str.title()
    log["txn_type_values_standardized"] = int((raw_types != df["txn_type"]).sum())
    log["unique_txn_types_after"] = sorted(df["txn_type"].unique().tolist())

    # 2. Parse mixed date formats ('YYYY-MM-DD' and 'YYYY/MM/DD')
    raw_dates = df["txn_date"].astype(str).str.strip()
    log["slash_dates_normalized"] = int(raw_dates.str.contains("/", regex=False).sum())
    df["txn_date"] = pd.to_datetime(raw_dates, format="mixed", errors="coerce")
    invalid_dates = df["txn_date"].isna()
    log["invalid_dates_dropped"] = int(invalid_dates.sum())
    df = df.loc[~invalid_dates].copy()

    # 3. Deduplicate: exact rows first, then by txn_id preferring valid agent_ids
    exact_dups = int(df.duplicated().sum())
    df = df.drop_duplicates().copy()
    log["exact_duplicate_rows_dropped"] = exact_dups

    valid_agents = set(agents_clean["agent_id"])
    df["_valid_agent"] = df["agent_id"].isin(valid_agents)
    df = df.sort_values(by=["txn_id", "_valid_agent"], ascending=[True, False])
    id_dups = int(df.duplicated(subset=["txn_id"], keep="first").sum())
    df = (
        df.drop_duplicates(subset=["txn_id"], keep="first")
          .drop(columns=["_valid_agent"])
          .copy()
    )
    log["duplicate_txn_ids_dropped"] = id_dups

    # 4. Coerce amount to numeric, drop missing/non-positive values, and remove extreme outliers
    df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
    missing_amt = df["amount"].isna() | (df["amount"] <= 0)
    log["missing_or_nonpositive_amounts_dropped"] = int(missing_amt.sum())
    df = df.loc[~missing_amt].copy()

    # Robust Modified Z-score on log(amount) to catch extreme entry errors without clipping natural skew
    log_amt = np.log(df["amount"])
    med_log = log_amt.median()
    mad_log = np.median(np.abs(log_amt - med_log))
    mod_z = 0.6745 * (log_amt - med_log) / mad_log
    outlier_mask = mod_z > 4.0
    log["extreme_outliers_dropped"] = int(outlier_mask.sum())
    log["outlier_values_removed"] = sorted(df.loc[outlier_mask, "amount"].tolist())
    df = df.loc[~outlier_mask].copy()

    # 5. Enforce referential integrity against agents master table & enrich
    orphan_mask = ~df["agent_id"].isin(valid_agents)
    log["orphan_agent_ids_found"] = sorted(df.loc[orphan_mask, "agent_id"].unique().tolist())
    log["orphan_agent_rows_dropped"] = int(orphan_mask.sum())
    df = df.loc[~orphan_mask].copy()

    clean_df = (
        df.merge(agents_clean, on="agent_id", how="inner")
          .sort_values("txn_id")
          .reset_index(drop=True)
    )
    log["final_clean_rows"] = int(len(clean_df))
    log["total_rows_removed"] = int(log["initial_txn_rows"] - log["final_clean_rows"])

    return clean_df, log


# Example usage:
agents_raw = pd.read_csv("agents.csv")
txns_raw = pd.read_csv("agent_transactions.csv")
clean_df, audit_log = clean_agent_data(txns_raw, agents_raw)

for key, val in audit_log.items():
    print(f"{key}: {val}")

initial_txn_rows: 612
initial_agent_rows: 20
txn_type_values_standardized: 384
unique_txn_types_after: ['Deposit', 'Withdrawal']
slash_dates_normalized: 5
invalid_dates_dropped: 0
exact_duplicate_rows_dropped: 11
duplicate_txn_ids_dropped: 1
missing_or_nonpositive_amounts_dropped: 20
extreme_outliers_dropped: 5
outlier_values_removed: [250000.0, 250000.0, 250000.0, 250000.0, 250000.0]
orphan_agent_ids_found: ['AG099']
orphan_agent_rows_dropped: 7
final_clean_rows: 568
total_rows_removed: 44
